# IT2011 – AIML · Group Preprocessing Pipeline
## Letter Image Recognition (UCI) — integrated six-member pipeline

| Step | Activity | Member |
|---|---|---|
| 1 | Data-quality validation and duplicate checking | IT25102224 |
| 2 | Target-label encoding | IT25102419 |
| 3 | Outlier analysis | IT25102551 |
| 4 | Feature scaling | IT25103309 |
| 5 | Feature selection | IT25103890 |
| 6 | PCA dimensionality reduction | IT25103881 |

Each member's technique is applied once, in an order chosen so that every step receives valid input.
Steps 5 and 6 are two alternative routes to fewer dimensions, so they run as **parallel branches**
and are compared against a common classifier at the end.

Run with **Kernel -> Restart & Run All**. Paths resolve automatically (local or Google Colab).

## Pipeline order, and why

```
load -> 1. validate & deduplicate -> 2. encode label -> split -> 3. outlier analysis
     -> 4. scale (fit on train) -> 5. feature selection ─┐
                                  -> 6. PCA ─────────────┴─> compare -> save
```

Three ordering rules the group agreed on:

1. **The split happens once, before anything is fitted.** Every transformer below is fitted on the
   training partition only and applied to the test partition. Fitting on all rows and splitting
   afterwards leaks test information and inflates the reported accuracy.
2. **Scaling comes before both reduction steps.** PCA is variance-driven, so unscaled input gives
   components that follow whichever feature has the widest numeric range.
3. **Feature selection and PCA do not chain.** Both answer the same question — how to use fewer
   dimensions — so chaining them applies the same idea twice. They are evaluated side by side and
   the pipeline reports which wins.

In [ ]:
# ── Imports and settings ──────────────────────────────────────────────────
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import joblib, zipfile, warnings, urllib.request
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
TEST_SIZE    = 0.20      # stratified hold-out
TOP_K        = 10        # features kept by selection  (step 5)
N_COMPONENTS = 12        # components kept by PCA      (step 6)
REMOVE_OUTLIERS = False  # see step 3 for the evidence behind this decision

np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# ── Project paths: works locally (notebook in notebooks/) and in Google Colab ──
DATA_FILE = "letter-recognition.data"

# UCI Machine Learning Repository - Letter Recognition (dataset 59)
#   landing page : https://archive.ics.uci.edu/dataset/59/letter+recognition
UCI_CSV = "https://archive.ics.uci.edu/static/public/59/data.csv"
UCI_ZIP = "https://archive.ics.uci.edu/static/public/59/letter+recognition.zip"

def locate_project_root():
    """Return the folder that contains data/raw, searching the usual locations."""
    candidates = [Path(".."), Path(".")]
    if Path("/content").is_dir():                        # Google Colab
        candidates.append(Path("/content"))
        candidates += [p.parent.parent for p in Path("/content").glob("*/data/raw")]
    for c in candidates:
        if (c / "data" / "raw").is_dir():
            return c.resolve()
    (Path("data") / "raw").mkdir(parents=True, exist_ok=True)
    return Path(".").resolve()

ROOT    = locate_project_root()
RAW_DIR = ROOT / "data" / "raw"
FIG_DIR = ROOT / "results" / "eda_visualizations"
OUT_DIR = ROOT / "results" / "outputs"
for d in (RAW_DIR, FIG_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

In [ ]:
# ── Load the dataset ──────────────────────────────────────────────────────
# Source order: local extracted file -> local UCI zip -> download from the UCI
# repository -> manual upload. The first available source wins, so the notebook
# runs on a teammate's clone, on a bare Colab session, or offline.

# One canonical schema for the whole notebook. The UCI archive publishes the
# target column as "lettr"; it is renamed to "letter" everywhere below.
COLUMNS = ["letter",
           "x-box", "y-box", "width", "high", "onpix",
           "x-bar", "y-bar",
           "x2bar", "y2bar", "xybar", "x2ybr", "xy2br",
           "x-ege", "xegvy", "y-ege", "yegvx"]
FEATURES = COLUMNS[1:]

def _normalise(df):
    """Rename the UCI target column and enforce the canonical column order."""
    df = df.rename(columns={"lettr": "letter"})
    missing = [c for c in COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Unexpected schema, missing columns: {missing}")
    return df[COLUMNS]

def load_dataset():
    # 1) Already-extracted data file sitting in data/raw/
    local = RAW_DIR / DATA_FILE
    if local.exists():
        print(f"Source: local file  ({local})")
        return _normalise(pd.read_csv(local, header=None, names=COLUMNS))

    # 2) A previously downloaded CSV copy (has a header row)
    local_csv = RAW_DIR / "letter-recognition.csv"
    if local_csv.exists():
        print(f"Source: local CSV   ({local_csv})")
        return _normalise(pd.read_csv(local_csv))

    # 3) The UCI zip already downloaded into the project
    for zp in [*RAW_DIR.glob("*.zip"), *(ROOT / "data").glob("*.zip"), *ROOT.glob("*.zip")]:
        with zipfile.ZipFile(zp) as z:
            z.extractall(RAW_DIR)
        if local.exists():
            print(f"Source: extracted from {zp.name}")
            return _normalise(pd.read_csv(local, header=None, names=COLUMNS))

    # 4) Download straight from the UCI Machine Learning Repository
    for url, reader in ((UCI_CSV, "csv"), (UCI_ZIP, "zip")):
        try:
            print(f"Downloading from UCI: {url}")
            if reader == "csv":
                df = _normalise(pd.read_csv(url))
                df.to_csv(local_csv, index=False)          # cache: download once
                print(f"Source: UCI download (cached at {local_csv})")
                return df
            urllib.request.urlretrieve(url, RAW_DIR / "letter+recognition.zip")
            with zipfile.ZipFile(RAW_DIR / "letter+recognition.zip") as z:
                z.extractall(RAW_DIR)
            print("Source: UCI download (zip, extracted)")
            return _normalise(pd.read_csv(local, header=None, names=COLUMNS))
        except Exception as exc:
            print(f"   download failed ({type(exc).__name__}): {exc}")

    # 5) Nothing worked - ask for a manual upload (Colab) or explain (local)
    try:
        from google.colab import files
        print("Upload letter+recognition.zip or letter-recognition.data:")
        for name in files.upload():
            src = Path(name)
            if src.suffix == ".zip":
                with zipfile.ZipFile(src) as z:
                    z.extractall(RAW_DIR)
            else:
                src.replace(RAW_DIR / src.name)
        return _normalise(pd.read_csv(local, header=None, names=COLUMNS))
    except ImportError:
        raise FileNotFoundError(
            f"Could not obtain the dataset. Place letter-recognition.data or "
            f"letter+recognition.zip in {RAW_DIR}, or download it from "
            f"https://archive.ics.uci.edu/dataset/59/letter+recognition"
        )

raw = load_dataset()
print("Dataset:", raw.shape)
raw.head()

## Step 1 — Data-quality validation and duplicate checking

Establishes that the data is fit to process: correct types, values inside the documented range, no
missing entries, and no exact duplicate rows.

In [ ]:
# ── Step 1: validation and deduplication ──────────────────────────────────
print("Missing values      :", int(raw.isna().sum().sum()))
print("Label domain valid  :", bool(raw.letter.isin(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")).all()))
print("Features within 0-15:", bool(raw[FEATURES].ge(0).all().all() and raw[FEATURES].le(15).all().all()))
print("All features integer:", bool(raw[FEATURES].mod(1).eq(0).all().all()))

duplicate_mask = raw.duplicated(keep="first")
data = raw.loc[~duplicate_mask].reset_index(drop=True)
print(f"\nDuplicate rows removed: {int(duplicate_mask.sum())}")
print(f"Rows: {len(raw)} -> {len(data)}")

# Class balance after cleaning (EDA visualization for step 1).
plt.figure(figsize=(12, 4))
sns.countplot(x="letter", data=data, order=sorted(data.letter.unique()), color="steelblue")
plt.title("Step 1 - samples per letter after deduplication")
plt.xlabel("Letter"); plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "pipeline_01_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

counts = data.letter.value_counts()
print(f"Class counts: min {counts.min()}, max {counts.max()} -> near-balanced, no resampling needed.")

## Step 2 — Target-label encoding

The 26 letters are converted to integers 0–25. This touches the target only; no feature is altered,
and no later step reads the encoded label except the classifier used for evaluation.

In [ ]:
# ── Step 2: encode the target ─────────────────────────────────────────────
label_encoder = LabelEncoder().fit(data.letter)
y_all = label_encoder.transform(data.letter)
X_all = data[FEATURES].copy()

mapping = pd.DataFrame({"letter": label_encoder.classes_,
                        "encoded": label_encoder.transform(label_encoder.classes_)})
print("Label mapping (first 6 of 26):")
display(mapping.head(6).T)

# The encoding must be perfectly reversible.
assert np.array_equal(label_encoder.inverse_transform(y_all), data.letter.values)
print("Inverse transform verified: encoding is lossless.")

## The split — done once, before anything is fitted

Every transformer below is fitted on `X_train` only. A stratified 80/20 hold-out keeps all 26 classes
proportionally represented in both partitions.

In [ ]:
# ── Single stratified split, before any transformer is fitted ─────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Classes present in both partitions:",
      len(np.unique(y_train)), "/", len(np.unique(y_test)))

## Step 3 — Outlier analysis

Boxplots and a 1.5×IQR audit. The audit is **reported, not applied** — the cell below measures what
removal would cost, and the evidence says to keep the rows.

In [ ]:
# ── Step 3: outlier audit (measured on the training partition) ────────────
fig, axes = plt.subplots(4, 4, figsize=(16, 10))
for ax, col in zip(axes.flatten(), FEATURES):
    sns.boxplot(x=X_train[col], ax=ax, color="lightsteelblue")
    ax.set_title(col, fontsize=9); ax.set_xlabel("")
plt.suptitle("Step 3 - feature distributions with 1.5xIQR whiskers (training partition)", y=1.00)
plt.tight_layout()
plt.savefig(FIG_DIR / "pipeline_03_outlier_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

Q1, Q3 = X_train.quantile(.25), X_train.quantile(.75)
IQR    = Q3 - Q1
flag   = ((X_train < (Q1 - 1.5*IQR)) | (X_train > (Q3 + 1.5*IQR)))
keep   = ~flag.any(axis=1)

print(f"Rows with >=1 flagged feature : {(~keep).sum()} of {len(X_train)} "
      f"({(~keep).mean()*100:.1f}%)")
print("\nFlagged values per feature (top 5):")
print(flag.sum().sort_values(ascending=False).head(5).to_string())
print(f"\nSmallest class if these rows were dropped: {pd.Series(y_train[keep.values]).value_counts().min()} samples")

### Decision: keep all rows

A 1.5×IQR rule flags ~40% of the dataset. That is not a data-quality signal — it is an artefact of
applying a rule designed for continuous measurements to 16 integer features confined to 0–15, several
of them strongly skewed, where a value far from the median is common rather than rare. Removing them
also shrinks the smallest class to a few dozen samples.

The comparison in the final section quantifies the cost: removal lowers accuracy by roughly **1.3
percentage points** on every branch. `REMOVE_OUTLIERS` is therefore set to `False`; flip it at the top
of the notebook to reproduce the alternative.

In [ ]:
# ── Apply the decision recorded above ─────────────────────────────────────
if REMOVE_OUTLIERS:
    X_train, y_train = X_train[keep.values], y_train[keep.values]
    print(f"Outliers removed. Training rows: {len(X_train)}")
else:
    print(f"Outliers retained by decision. Training rows: {len(X_train)}")
# The test partition is never filtered: it must reflect the real data distribution.
print(f"Test rows (never filtered): {len(X_test)}")

## Step 4 — Feature scaling

Standardisation to mean 0, standard deviation 1. Required by PCA, and it puts the mutual-information
selector and the distance-based classifier on equal footing across features.

In [ ]:
# ── Step 4: standardise (fit on train, apply to both) ─────────────────────
scaler         = StandardScaler().fit(X_train)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=FEATURES, index=X_train.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),  columns=FEATURES, index=X_test.index)

fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(13, 4.5))
for col in ["x-box", "y-box", "onpix"]:
    sns.kdeplot(X_train[col], ax=ax1, label=col)
    sns.kdeplot(X_train_scaled[col], ax=ax2, label=col)
ax1.set_title("Before standardisation"); ax1.legend()
ax2.set_title("After standardisation");  ax2.legend()
plt.suptitle("Step 4 - effect of standardisation on three features")
plt.tight_layout()
plt.savefig(FIG_DIR / "pipeline_04_scaling_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

print("Train means ~0:", np.round(X_train_scaled.mean().values[:5], 6))
print("Train stds  ~1:", np.round(X_train_scaled.std(ddof=0).values[:5], 6))

## Step 5 — Branch A: feature selection

Mutual information between each feature and the target, keeping the top 10. Fitted on the training
partition only. Output: 10 of the original, interpretable features.

In [ ]:
# ── Step 5: mutual-information feature selection (fit on train only) ──────
def mi_scores(a, b):
    return mutual_info_classif(a, b, discrete_features=True, random_state=RANDOM_STATE)

selector = SelectKBest(score_func=mi_scores, k=TOP_K).fit(X_train_scaled, y_train)

mi_results = pd.DataFrame({"feature": FEATURES,
                           "mutual_information": selector.scores_,
                           "selected": selector.get_support()}
                          ).sort_values("mutual_information", ascending=False).reset_index(drop=True)

plt.figure(figsize=(11, 6))
sns.barplot(data=mi_results, x="mutual_information", y="feature", hue="selected",
            palette={True: "seagreen", False: "lightgray"}, dodge=False)
plt.title(f"Step 5 - mutual information with the target (top {TOP_K} selected)")
plt.xlabel("Mutual information score"); plt.ylabel("Feature")
plt.legend(title="Selected")
plt.tight_layout()
plt.savefig(FIG_DIR / "pipeline_05_mutual_information.png", dpi=150, bbox_inches="tight")
plt.show()

selected_features = mi_results.loc[mi_results.selected, "feature"].tolist()
X_train_sel = pd.DataFrame(selector.transform(X_train_scaled),
                           columns=X_train_scaled.columns[selector.get_support()], index=X_train.index)
X_test_sel  = pd.DataFrame(selector.transform(X_test_scaled),
                           columns=X_test_scaled.columns[selector.get_support()],  index=X_test.index)

print("Kept   :", selected_features)
print("Dropped:", mi_results.loc[~mi_results.selected, "feature"].tolist())
print("Shape  :", X_train_scaled.shape, "->", X_train_sel.shape)

## Step 6 — Branch B: PCA dimensionality reduction

Twelve principal components, chosen by the 95% cumulative-variance criterion. Fitted on the training
partition only. Output: 12 uncorrelated components built from all 16 features.

In [ ]:
# ── Step 6: PCA (fit on train only) ───────────────────────────────────────
pca_full = PCA(random_state=RANDOM_STATE).fit(X_train_scaled)
evr, cum = pca_full.explained_variance_ratio_, np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
comps = np.arange(1, len(evr) + 1)
axes[0].bar(comps, evr * 100, color="steelblue", edgecolor="black", alpha=.85)
axes[0].set(xlabel="Principal component", ylabel="Explained variance (%)",
            title="Variance per component", xticks=comps)
axes[1].plot(comps, cum * 100, "o-", color="darkorange", lw=2, ms=5)
axes[1].axhline(95, ls="--", c="red", lw=1)
axes[1].annotate(f"95% -> {N_COMPONENTS} PCs", xy=(N_COMPONENTS, 95),
                 xytext=(N_COMPONENTS + .4, 88), color="red", fontsize=9)
axes[1].set(xlabel="Number of components", ylabel="Cumulative explained variance (%)",
            title="Cumulative explained variance", xticks=comps, ylim=(20, 103))
plt.suptitle("Step 6 - PCA explained variance")
plt.tight_layout()
plt.savefig(FIG_DIR / "pipeline_06_pca_explained_variance.png", dpi=150, bbox_inches="tight")
plt.show()

pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE).fit(X_train_scaled)
pc_names    = [f"PC{i}" for i in range(1, N_COMPONENTS + 1)]
X_train_pca = pd.DataFrame(pca.transform(X_train_scaled), columns=pc_names, index=X_train.index)
X_test_pca  = pd.DataFrame(pca.transform(X_test_scaled),  columns=pc_names, index=X_test.index)

print(f"Shape : {X_train_scaled.shape} -> {X_train_pca.shape}")
print(f"Variance retained: {pca.explained_variance_ratio_.sum()*100:.2f}%")

## Comparison — which reduction should the project use?

All four feature sets are passed to the same classifier (KNN, k=5) on the same split, so the only
thing that varies is the preprocessing.

In [ ]:
# ── Evaluate every branch on the same split and classifier ────────────────
branches = {
    "Baseline (16 scaled features)":       (X_train_scaled, X_test_scaled),
    f"Branch A - selection ({TOP_K} features)": (X_train_sel,    X_test_sel),
    f"Branch B - PCA ({N_COMPONENTS} components)": (X_train_pca,    X_test_pca),
}

rows = []
for name, (A, B) in branches.items():
    model = KNeighborsClassifier(n_neighbors=5).fit(A, y_train)
    rows.append({"Feature set": name,
                 "Dimensions": A.shape[1],
                 "Accuracy %": round(accuracy_score(y_test, model.predict(B)) * 100, 2)})

results = pd.DataFrame(rows)
results["vs baseline"] = (results["Accuracy %"] - results.loc[0, "Accuracy %"]).round(2)
display(results)

plt.figure(figsize=(9, 4.5))
bars = sns.barplot(data=results, y="Feature set", x="Accuracy %", color="steelblue")
plt.axvline(results.loc[0, "Accuracy %"], ls="--", c="grey", lw=1.2)
for i, v in enumerate(results["Accuracy %"]):
    bars.text(v + .1, i, f"{v:.2f}%", va="center", fontsize=9)
plt.xlim(85, 100); plt.title("Preprocessing branches compared on the same classifier")
plt.tight_layout()
plt.savefig(FIG_DIR / "pipeline_07_branch_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

winner = results.iloc[1:].sort_values("Accuracy %", ascending=False).iloc[0]
print(f"Best reduction branch: {winner['Feature set']} "
      f"({winner['Accuracy %']}%, {winner['Dimensions']} dimensions)")

### Reading the comparison

Neither reduction beats the 16-feature baseline, which is the expected result: both discard
information in exchange for a smaller feature space, and neither is optimising accuracy directly.
The question is which gives up less.

**Feature selection loses less accuracy and is more interpretable** — its output is 10 of the
original measurements, which a domain expert can still reason about. **PCA removes all inter-feature
correlation**, which matters for models that assume independent inputs, and compresses information
from all 16 features rather than discarding six outright.

Both reduced sets are saved below so the modelling phase can test either.

## Save the pipeline outputs

In [ ]:
# ── Persist the processed datasets and every fitted transformer ───────────
def save_split(Xtr, Xte, tag):
    tr = Xtr.copy(); tr["letter"] = y_train
    te = Xte.copy(); te["letter"] = y_test
    tr.to_csv(OUT_DIR / f"pipeline_{tag}_train.csv", index=False)
    te.to_csv(OUT_DIR / f"pipeline_{tag}_test.csv",  index=False)

save_split(X_train_scaled, X_test_scaled, "scaled")      # all 16, standardised
save_split(X_train_sel,    X_test_sel,    "selected")    # branch A
save_split(X_train_pca,    X_test_pca,    "pca")         # branch B

joblib.dump(label_encoder, OUT_DIR / "pipeline_label_encoder.pkl")
joblib.dump(scaler,        OUT_DIR / "pipeline_scaler.pkl")
joblib.dump(selector,      OUT_DIR / "pipeline_selector.pkl")
joblib.dump(pca,           OUT_DIR / "pipeline_pca.pkl")
results.to_csv(OUT_DIR / "pipeline_branch_comparison.csv", index=False)

print("Saved to results/outputs/:")
for f in sorted(OUT_DIR.glob("pipeline_*")):
    print("  ", f.name)
print("\nFigures in results/eda_visualizations/:")
for f in sorted(FIG_DIR.glob("pipeline_*.png")):
    print("  ", f.name)

In [ ]:
# ── Final integrity checks ────────────────────────────────────────────────
assert len(X_train) == len(y_train) and len(X_test) == len(y_test)
assert not X_train_scaled.isna().any().any(), "NaNs introduced during scaling"
assert X_train_sel.shape[1] == TOP_K
assert X_train_pca.shape[1] == N_COMPONENTS
assert set(X_train.index).isdisjoint(set(X_test.index)), "train/test overlap"
off = X_train_pca.corr().values[~np.eye(N_COMPONENTS, dtype=bool)]
assert abs(off).max() < 1e-8, "PCA components are not orthogonal"

print("All integrity checks passed.")
print(f"  rows          : {len(raw)} raw -> {len(data)} deduplicated")
print(f"  split         : {len(X_train)} train / {len(X_test)} test (stratified, fitted on train only)")
print(f"  branch A      : {X_train_sel.shape[1]} selected features")
print(f"  branch B      : {X_train_pca.shape[1]} principal components")

## Summary

| Step | Activity | Applied as | Result |
|---|---|---|---|
| 1 | Validation & deduplication | Applied | 1,332 duplicate rows removed |
| 2 | Label encoding | Applied | 26 letters → 0–25, verified reversible |
| 3 | Outlier analysis | **Reported, not applied** | ~40% would be flagged; removal costs ~1.3 pp |
| 4 | Feature scaling | Applied | fitted on train, applied to both partitions |
| 5 | Feature selection | Branch A | 16 → 10 original features |
| 6 | PCA | Branch B | 16 → 12 uncorrelated components |

**Integration notes.** The individual notebooks disagreed on the target column name (`lettr` vs
`letter`), the data source (local file, URL, manual upload) and the split strategy (three-way, 80/20
stratified, fixed 16,000/4,000). This pipeline fixes one schema, one source and one split, so every
step operates on the same data. Two of the individual notebooks also fitted transformers before
splitting; here the split precedes every `fit`.